# Day 058 — Exercise 1: SSE Counting Stream

**Server-Sent Events (SSE)** let the server push data to the client over a single HTTP connection. FastAPI's `StreamingResponse` with `media_type='text/event-stream'` implements SSE. Each event is a `'data: ...\n\n'` string (two newlines = event delimiter).

TestClient buffers the full response body, so you can assert on it as a plain string.

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from starlette.testclient import TestClient


## Task

Implement `build_sse_api()` — return a FastAPI app with:

```
GET /count?n=5
```

- Streams n SSE events: `data: 0\n\n`, `data: 1\n\n`, …, `data: {n-1}\n\n`
- Ends with `data: [DONE]\n\n`
- Returns `Content-Type: text/event-stream`

## Your Implementation

In [ ]:
def build_sse_api() -> FastAPI:
    """Return a FastAPI app with GET /count?n=5 that streams SSE events.

    Each event:   'data: {i}\\n\\n' for i in range(n)
    Final event:  'data: [DONE]\\n\\n'
    Content-Type: text/event-stream
    """
    # TODO: create app, add GET /count route, return StreamingResponse
    raise NotImplementedError


In [ ]:
def build_sse_api() -> FastAPI:
    app = FastAPI()

    @app.get("/count")
    def stream_count(n: int = 5):
        def generate():
            for i in range(n):
                yield "data: " + str(i) + "\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(generate(), media_type="text/event-stream")

    return app


## Automated checks

In [ ]:
score, total = 0, 4
try:
    app    = build_sse_api()
    client = TestClient(app, raise_server_exceptions=False)

    r = client.get("/count?n=3")
    assert r.status_code == 200, f"Expected 200, got {r.status_code}"
    score += 1; print("\u2705 /count returns 200")

    ct = r.headers.get("content-type", "")
    assert "text/event-stream" in ct, f"Expected text/event-stream, got {ct}"
    score += 1; print("\u2705 content-type is text/event-stream")

    assert "data: 0" in r.text and "data: 1" in r.text and "data: 2" in r.text, (
        f"Body missing expected data lines: {r.text!r}")
    score += 1; print("\u2705 body contains SSE data lines")

    assert "[DONE]" in r.text, f"[DONE] sentinel missing from body"
    score += 1; print("\u2705 [DONE] sentinel present")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_sse_api() -> FastAPI:
    app = FastAPI()

    @app.get("/count")
    def stream_count(n: int = 5):
        def generate():
            for i in range(n):
                yield "data: " + str(i) + "\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(generate(), media_type="text/event-stream")

    return app
```

**Why it works:** `StreamingResponse` accepts any Python generator. Each `yield` immediately flushes one SSE event to the client. The `[DONE]` sentinel is a convention so the browser `EventSource` knows when to stop listening.

</details>